In [2]:
"""
LatentGEE 단계적 테스트 스크립트
main() 실행 전 이 파일을 먼저 실행해서 파이프라인 전체를 검증합니다.
기존 코드(latentgee.py)와 같은 디렉토리에 두고 실행하세요.
"""

import torch
import numpy as np
import optuna
import traceback

# ── 기존 모듈에서 import (파일명에 맞게 수정)
from prototype import (
    VAE, train_vae, evaluate_latentgee,
    set_seed, load_hivrc, save_dated_filename
)

optuna.logging.set_verbosity(optuna.logging.WARNING)  # Optuna 로그 최소화


# ──────────────────────────────────────────────
# 공통 설정
# ──────────────────────────────────────────────
DATA_PATH  = "F:/졸업 후 연구/LatentGEE/Data"   # ← 실제 경로로 수정
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_tensor():
    X_raw, dat_merged = load_hivrc(DATA_PATH)

    # NaN 확인
    n_nan = X_raw.isna().sum().sum()
    if n_nan > 0:
        print(f"  ⚠️  NaN {n_nan}개 발견 → 0으로 대체")
        X_raw = X_raw.fillna(0)
    else:
        print(f"  ✅ NaN 없음")

    X_tensor = torch.tensor(X_raw.values, dtype=torch.float32).to(DEVICE)
    print(f"  X_tensor shape : {X_tensor.shape}")
    print(f"  값 범위        : [{X_tensor.min():.2f}, {X_tensor.max():.2f}]")
    print(f"  0 비율         : {(X_tensor == 0).float().mean():.2%}")
    return X_tensor, X_raw, dat_merged


# ──────────────────────────────────────────────
# 1단계: 데이터 로드 & NaN 확인
# ──────────────────────────────────────────────
def test_stage1_data():
    print("\n" + "="*50)
    print("1단계: 데이터 로드 & NaN 확인")
    print("="*50)
    try:
        X_tensor, X_raw, dat_merged = load_tensor()
        print(f"  dat_merged shape: {dat_merged.shape}")
        print("  ✅ 1단계 통과\n")
        return X_tensor, X_raw, dat_merged
    except Exception as e:
        print(f"  ❌ 1단계 실패: {e}")
        traceback.print_exc()
        return None, None, None


# ──────────────────────────────────────────────
# 2단계: VAE forward pass (shape/device 오류 확인)
# ──────────────────────────────────────────────
def test_stage2_forward(X_tensor):
    print("="*50)
    print("2단계: VAE forward pass")
    print("="*50)
    try:
        set_seed()
        model = VAE(
            input_dim   = X_tensor.shape[1],
            latent_dim  = 16,
            n_layers    = 2,
            base_dim    = 256,
        ).to(DEVICE)

        with torch.no_grad():
            (pi, mu_x, log_sigma_x), mu_z, logvar_z, z = model(X_tensor[:8])  # 8개 샘플만

        print(f"  pi shape       : {pi.shape}")
        print(f"  mu_x shape     : {mu_x.shape}")
        print(f"  z shape        : {z.shape}")
        print(f"  pi 범위        : [{pi.min():.4f}, {pi.max():.4f}]  (0~1 이어야 함)")
        print(f"  NaN 여부       : pi={pi.isnan().any()}, mu={mu_x.isnan().any()}, z={z.isnan().any()}")
        print("  ✅ 2단계 통과\n")
        return model
    except Exception as e:
        print(f"  ❌ 2단계 실패: {e}")
        traceback.print_exc()
        return None


# ──────────────────────────────────────────────
# 3단계: 단일 모델 학습 (5 epoch)
# ──────────────────────────────────────────────
def test_stage3_train(X_tensor):
    print("="*50)
    print("3단계: 단일 모델 학습 (5 epoch)")
    print("="*50)
    try:
        set_seed()
        model = VAE(
            input_dim   = X_tensor.shape[1],
            latent_dim  = 16,
            n_layers    = 2,
            base_dim    = 256,
        ).to(DEVICE)

        loss = train_vae(model, X_tensor,
                         beta=1.0, epochs=5, lr=1e-3,
                         batch_size=64, weight_decay=1e-5,
                         grad_clip_norm=1.0)

        print(f"  최종 loss: {loss:.4f}")
        if np.isnan(loss):
            print("  ❌ loss가 NaN — learning rate를 낮추거나 데이터 스케일 확인 필요")
            return None
        print("  ✅ 3단계 통과\n")
        return model
    except Exception as e:
        print(f"  ❌ 3단계 실패: {e}")
        traceback.print_exc()
        return None


# ──────────────────────────────────────────────
# 4단계: evaluate_latentgee (HDBSCAN + GEE + silhouette)
# ──────────────────────────────────────────────
def test_stage4_evaluate(model, X_tensor):
    print("="*50)
    print("4단계: evaluate_latentgee")
    print("="*50)
    try:
        labels, sil, noise_ratio = evaluate_latentgee(
            model, X_tensor,
            min_cluster_size=10,
            min_samples=None,
            metric="euclidean",
            cluster_selection_method="eom"
        )
        n_clusters = len(np.unique(labels[labels != -1]))
        print(f"  클러스터 수  : {n_clusters}")
        print(f"  노이즈 비율  : {noise_ratio:.2%}")
        print(f"  silhouette   : {sil:.4f}")

        if n_clusters < 2:
            print("  ⚠️  클러스터가 1개 이하 — min_cluster_size를 낮추거나 latent_dim 조정 필요")
        if noise_ratio > 0.5:
            print("  ⚠️  노이즈 비율이 50% 초과 — 학습이 충분하지 않을 수 있음")
        print("  ✅ 4단계 통과\n")
        return True
    except Exception as e:
        print(f"  ❌ 4단계 실패: {e}")
        traceback.print_exc()
        return False


# ──────────────────────────────────────────────
# 5단계: Optuna 5 trial (전체 흐름 확인)
# ──────────────────────────────────────────────
def test_stage5_optuna(X_tensor):
    print("="*50)
    print("5단계: Optuna 5 trial")
    print("="*50)

    import yaml
    CONFIG_PATH = "C:/Users/KOBIC/Documents/latentgee/examples/config.yaml"
    from latentgee import objective, save_dated_filename

    try:
        with open(CONFIG_PATH) as f:
            cfg = yaml.safe_load(f)

        log_file = save_dated_filename("optuna_test", ".csv")

        study = optuna.create_study(
            direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=42)
        )
        study.optimize(
            lambda tr: objective(tr, cfg, X_tensor, log_file=log_file),
            n_trials=5,
        )

        completed = [t for t in study.trials
                     if t.state == optuna.trial.TrialState.COMPLETE]
        pruned    = [t for t in study.trials
                     if t.state == optuna.trial.TrialState.PRUNED]

        print(f"  완료 trial : {len(completed)} / 5")
        print(f"  pruned     : {len(pruned)} / 5")

        if completed:
            print(f"  best sil   : {study.best_value:.4f}")
            print(f"  best params: {study.best_params}")

        if len(completed) == 0:
            print("  ⚠️  완료된 trial이 없음 — 모든 trial이 pruned됨, 탐색 범위 재검토 필요")
        else:
            print("  ✅ 5단계 통과\n")

    except Exception as e:
        print(f"  ❌ 5단계 실패: {e}")
        traceback.print_exc()


# ──────────────────────────────────────────────
# 전체 실행
# ──────────────────────────────────────────────
if __name__ == "__main__":
    print("LatentGEE 파이프라인 단계별 테스트 시작")
    print(f"Device: {DEVICE}\n")

    # 1단계
    X_tensor, X_raw, dat_merged = test_stage1_data()
    if X_tensor is None:
        print("1단계 실패 — 이후 테스트 중단")
        exit(1)

    # 2단계
    model = test_stage2_forward(X_tensor)
    if model is None:
        print("2단계 실패 — 이후 테스트 중단")
        exit(1)

    # 3단계
    model = test_stage3_train(X_tensor)
    if model is None:
        print("3단계 실패 — 이후 테스트 중단")
        exit(1)

    # 4단계
    ok = test_stage4_evaluate(model, X_tensor)
    if not ok:
        print("4단계 실패 — 5단계 Optuna는 건너뜀")
        exit(1)

    # 5단계
    test_stage5_optuna(X_tensor)

    print("="*50)
    print("모든 단계 완료. main() 실행 가능 상태입니다.")
    print("="*50)

ModuleNotFoundError: No module named 'torch'